# Stable Diffusion — Architecture & Fine-Tuning Notebook

> Hands-on Build It and Exercises.

## Build It

This lesson uses `diffusers` end-to-end rather than rebuilding Stable Diffusion from scratch. The pieces you would need to rebuild (VAE, text encoder, U-Net, scheduler) are topics of their own lessons; here the goal is fluency with the production API.

### Step 1: Text-to-image

In [ ]:
```python

import torch

from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(

    "runwayml/stable-diffusion-v1-5",

    torch_dtype=torch.float16,

).to("cuda")

image = pipe(

    prompt="a dog riding a skateboard in tokyo, studio ghibli style",

    guidance_scale=7.5,

    num_inference_steps=25,

    generator=torch.Generator("cuda").manual_seed(42),

).images[0]

image.save("dog.png")

In [ ]:
```

`float16` halves VRAM with no visible quality loss. `num_inference_steps=25` with the default DPM-Solver++ matches `num_inference_steps=50` with DDIM.

### Step 2: Swap the scheduler

In [ ]:
```python

from diffusers import DPMSolverMultistepScheduler, EulerAncestralDiscreteScheduler

pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)

In [ ]:
```

Scheduler state is decoupled from U-Net weights. You can train on DDPM and sample with any scheduler.

### Step 3: Image-to-image

In [ ]:
```python

from diffusers import StableDiffusionImg2ImgPipeline

from PIL import Image

img2img = StableDiffusionImg2ImgPipeline.from_pretrained(

    "runwayml/stable-diffusion-v1-5",

    torch_dtype=torch.float16,

).to("cuda")

init_image = Image.open("dog.png").convert("RGB").resize((512, 512))

out = img2img(

    prompt="a dog riding a skateboard, oil painting",

    image=init_image,

    strength=0.6,

    guidance_scale=7.5,

).images[0]

In [ ]:
```

`strength` is how much noise to add before denoising (0.0 = unchanged, 1.0 = full regeneration). 0.5-0.7 is the standard range for style transfer.

### Step 4: Inpainting

In [ ]:
```python

from diffusers import StableDiffusionInpaintPipeline

inpaint = StableDiffusionInpaintPipeline.from_pretrained(

    "runwayml/stable-diffusion-inpainting",

    torch_dtype=torch.float16,

).to("cuda")

image = Image.open("dog.png").convert("RGB").resize((512, 512))

mask = Image.open("dog_mask.png").convert("L").resize((512, 512))

out = inpaint(

    prompt="a cat",

    image=image,

    mask_image=mask,

    guidance_scale=7.5,

).images[0]

In [ ]:
```

White pixels in the mask are the area to regenerate. Black pixels are preserved.

### Step 5: LoRA loading

In [ ]:
```python

pipe.load_lora_weights("sayakpaul/sd-lora-ghibli")

pipe.fuse_lora(lora_scale=0.8)

image = pipe(prompt="a village square in ghibli style").images[0]

In [ ]:
```

`lora_scale` controls strength; 0.0 = no effect, 1.0 = full effect. `fuse_lora` bakes the adapter into the weights in place for speed, but prevents swapping. Call `pipe.unfuse_lora()` before loading a different adapter.

### Step 6: LoRA training (sketch)

Real LoRA training lives in `peft` or `diffusers.training`. The outline:

In [ ]:
```python

# Pseudocode

for step, batch in enumerate(dataloader):

    images, prompts = batch

    latents = vae.encode(images).latent_dist.sample() * 0.18215

    t = torch.randint(0, num_train_timesteps, (batch_size,))

    noise = torch.randn_like(latents)

    noisy_latents = scheduler.add_noise(latents, noise, t)

    text_emb = text_encoder(tokenizer(prompts))

    pred_noise = unet(noisy_latents, t, text_emb)  # LoRA weights injected here

    loss = F.mse_loss(pred_noise, noise)

    loss.backward()

    optimizer.step()

In [ ]:
```

Only the LoRA matrices receive gradient; the base U-Net, VAE, and text encoder are frozen. With a batch size of 1 and gradient checkpointing this fits in 8 GB of VRAM.

## Exercises

In [ ]:
1. **(Easy)** Generate the same prompt with `guidance_scale` in `[1, 3, 5, 7.5, 10, 15]`. Describe how the image changes. At what guidance value do artefacts appear?
2. **(Medium)** Take any real photograph, run it through `StableDiffusionImg2ImgPipeline` at `strength` in `[0.2, 0.4, 0.6, 0.8, 1.0]`. Which strength preserves composition while changing style? Why does 1.0 ignore the input entirely?
3. **(Hard)** Train a LoRA on 10-20 images of a single subject (a pet, a logo, a character) and generate novel scenes with that subject in them. Report the LoRA rank and training steps that produced the best identity preservation without overfitting to the input images.